# TP 2 : Expressions Régulières Avancées en Python
**Pr. Zouhair ELAMRANI ABOU ELASSAD**

Ce notebook contient les solutions complètes et détaillées des 4 exercices du TP :
1. **Exercice 1** : Analyse de logs serveur (`exo1_server.log`)
2. **Exercice 2** : Validation et normalisation d'un annuaire de contacts (`exo2_contacts.txt`)
3. **Exercice 3** : Nettoyage et transformation d'un article brut (`exo3_article.html`)
4. **Exercice 4** : Extraction de blocs de texte vers des fichiers CSV (`exo4_orders.txt`)


---
## Exercice 1 : Analyse de logs serveur (`exo1_server.log`)
### Objectifs :
- Lecture du fichier ligne par ligne.
- Expression régulière avec `re.VERBOSE` et groupes nommés (`date`, `time`, `level`, `ip`, `message`).
- Normalisation du niveau en majuscules.
- Validation stricte IPv4 (0-255 sans zéros non significatifs).
- Comptage via `collections.Counter`.
- Extraction des identifiants (`query_id=...`, `code=...`) et calcul de la durée moyenne des timeouts.
- Rapport final propre.


In [1]:
import re
from collections import Counter

# 1. Lecture du fichier
with open("./files/exo1_server.log", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

print(f"{len(raw_lines)} lines read")

# 2. Motif VERBOSE avec groupes nommés
LOG_PATTERN = re.compile(
    r"""
    ^(?P<date>\d{4}-\d{2}-\d{2})           # Date ISO (YYYY-MM-DD)
    \s+
    (?P<time>\d{2}:\d{2}:\d{2})           # Heure (HH:MM:SS)
    \s+
    \[(?P<level>[A-Za-z]+)\]               # Niveau de log insensible à la casse
    \s+
    (?P<ip>\S+)                             # Adresse IP brute
    \s+-\s+
    (?P<message>.*)$                         # Message
    """,
    re.VERBOSE | re.IGNORECASE
)

entries = []
bad_lines = []

for idx, line in enumerate(raw_lines, start=1):
    stripped = line.strip()
    match = LOG_PATTERN.match(stripped)
    if match:
        # 3. Normalisation du niveau en majuscules
        entry = match.groupdict()
        entry["level"] = entry["level"].upper()
        entries.append(entry)
    else:
        bad_lines.append((idx, stripped))

print(f"{len(entries)} entries, {len(bad_lines)} bad line(s):")
print(bad_lines)

# 4. Validation IPv4 stricte
# 0-255 sans zéros non significatifs: (25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)
OCTET = r"(?:25[0-5]|2[0-4]\d|1\d{2}|[1-9]?\d)"
IPV4 = re.compile(rf"^{OCTET}\.{OCTET}\.{OCTET}\.{OCTET}$")

valid_ip_entries = []
invalid_ips = []

for entry in entries:
    ip = entry["ip"]
    if IPV4.fullmatch(ip):
        valid_ip_entries.append(entry)
    else:
        invalid_ips.append(ip)

print(f"Invalid IPs: {sorted(invalid_ips)}")

# 5. Comptage avec Counter
# Nombre d'entrées ERROR par IP valide
error_ips = [entry["ip"] for entry in valid_ip_entries if entry["level"] == "ERROR"]
error_ip_counter = Counter(error_ips)

# Nombre d'entrées par niveau (sur l'ensemble des entrées parsées)
level_counter = Counter(entry["level"] for entry in entries)

print(error_ip_counter)
print(level_counter)

# 6. Extraction des identifiants et calcul du timeout moyen
# Extraction des identifiants (query_id=... ou code=...)
all_error_messages = " ".join(entry["message"] for entry in entries if entry["level"] == "ERROR")
identifiers = re.findall(r"(query_id|code)=([A-Z]-\d+)", all_error_messages)

# Recherche des durées de timeout
timeout_durations = [int(m.group(1)) for m in re.finditer(r"Database timeout after (\d+)s", all_error_messages)]
avg_timeout = sum(timeout_durations) / len(timeout_durations) if timeout_durations else 0.0

# 7. Affichage du rapport complet
print("=== LOG REPORT ===")
print(f"Parsed entries     : {len(entries)}")
print(f"Bad lines          : {[num for num, _ in bad_lines]}")
print(f"Entries invalid IP : {len(invalid_ips)} -> {sorted(invalid_ips)}")
print("Errors per valid IP:")
for ip, count in error_ip_counter.items():
    print(f"  {ip:<15} {count}")
print(f"Identifiers        : {identifiers}")
print(f"Average timeout    : {avg_timeout:.1f}s over {len(timeout_durations)} timeouts")


14 lines read
13 entries, 1 bad line(s):
[(8, 'corrupted line without proper format')]
Invalid IPs: ['010.0.0.7', '256.1.1.1', '999.12.1.4']
Counter({'192.168.1.23': 2, '10.0.0.5': 2, '172.16.4.8': 1})
Counter({'ERROR': 6, 'INFO': 3, 'WARNING': 2, 'DEBUG': 1, 'CRITICAL': 1})
=== LOG REPORT ===
Parsed entries     : 13
Bad lines          : [8]
Entries invalid IP : 3 -> ['010.0.0.7', '256.1.1.1', '999.12.1.4']
Errors per valid IP:
  192.168.1.23    2
  10.0.0.5        2
  172.16.4.8      1
Identifiers        : [('query_id', 'Q-4521'), ('query_id', 'Q-4533'), ('code', 'E-107'), ('query_id', 'Q-4602'), ('code', 'E-210'), ('code', 'E-300')]
Average timeout    : 29.0s over 3 timeouts


---
## Exercice 2 : Validation et normalisation d'un annuaire de contacts (`exo2_contacts.txt`)
### Objectifs :
- Extraction des noms via `re.findall` avec lookbehind et lookahead sans capturer les libellés.
- Validation stricte de l'adresse email (`re.fullmatch`).
- Normalisation du numéro de téléphone marocain (+212, 00212, 0 -> format `+212XXXXXXXXX`) via `re.sub` et groupe nommé.
- Liste des contacts valides et rejetés avec motif.
- Masquage des adresses email (`re.sub` avec fonction callback : `a***@univ-cadi.ac.ma`).
- Comptage des domaines via lookbehind sur `@` et `Counter`.


In [2]:
import re
from collections import Counter

# 1. Lecture globale et par lignes non vides
with open("./files/exo2_contacts.txt", "r", encoding="utf-8") as f:
    full_text = f.read()

with open("./files/exo2_contacts.txt", "r", encoding="utf-8") as f:
    non_empty_lines = [line.strip() for line in f.readlines() if line.strip()]

# 2. Extraction des noms avec un lookbehind et un lookahead
names = re.findall(r"(?<=Name:\s).*?(?=\s*\|)", full_text)
print(f"{len(names)} contacts")
print(names)

# 3. Motif Email conforme aux spécifications
# Local part: lettres/chiffres séparés par un seul ., _ ou - (pas de séparateur au début/fin)
# Domain: étiquettes séparées par des points, extension finale >= 2 lettres
email_pattern = re.compile(
    r"""
    ^[A-Za-z0-9]+                 # Début par un caractère alphanumérique
    ([._-][A-Za-z0-9]+)*          # Séparateur unique suivi d'alphanumérique
    @
    [A-Za-z0-9-]+                 # Nom d'hôte
    (\.[A-Za-z0-9-]+)*           # Sous-domaines facultatifs
    \.[A-Za-z]{2,}$               # Extension d'au moins 2 lettres
    """,
    re.VERBOSE
)

# 4 & 5. Normalisation téléphone et validation complète
phone_extract = re.compile(r"(?:Phone|Tel):\s*(.*)")
phone_validate = re.compile(r"^(?:\+212|00212|0)(?P<mobile>[67]\d{8})$")

valid_contacts = []
rejected_contacts = []

for line in non_empty_lines:
    # Extraction nom, email, tel
    name_m = re.search(r"Name:\s*(.*?)\s*\|", line)
    email_m = re.search(r"Email:\s*(.*?)\s*\|", line)
    phone_m = phone_extract.search(line)

    name = name_m.group(1).strip() if name_m else ""
    email_raw = email_m.group(1).strip() if email_m else ""
    phone_raw = phone_m.group(1).strip() if phone_m else ""

    reasons = []

    # Validation email
    if not email_pattern.fullmatch(email_raw):
        reasons.append("invalid email")

    # Nettoyage et validation téléphone
    cleaned_phone = re.sub(r"[\s.\-()]", "", phone_raw)
    phone_match = phone_validate.fullmatch(cleaned_phone)

    if not phone_match:
        reasons.append("invalid phone")
        normalized_phone = cleaned_phone
    else:
        normalized_phone = f"+212{phone_match.group('mobile')}"

    if not reasons:
        valid_contacts.append({
            "name": name,
            "email": email_raw,
            "phone": normalized_phone
        })
    else:
        rejected_contacts.append(f"{name} -> {', '.join(reasons)}")

print("VALID")
for c in valid_contacts:
    print(f"   {c}")

print("REJECTED")
for r in rejected_contacts:
    print(f"   {r}")

# 6. Masquage de l'email avec fonction de callback
def mask_email(match):
    first_char = match.group(1)
    domain_part = match.group(2)
    return f"{first_char}***@{domain_part}"

for c in valid_contacts:
    masked = re.sub(r"^([^@])[^@]*@(.*)$", mask_email, c["email"])
    print(f"{c['name']:<16} {masked}")

# 7. Comptage des domaines avec lookbehind sur @
domains = []
for c in valid_contacts:
    domain_match = re.search(r"(?<=@).*", c["email"])
    if domain_match:
        domains.append(domain_match.group(0))

domain_counter = Counter(domains)
for domain, count in domain_counter.most_common():
    print(f"{domain:<16} {count}")


9 contacts
['Amine El Fassi', 'Salma Benali', 'John Smith', 'Yassine Idrissi', "Lina O'Brien", 'Bad Entry', 'Omar Tazi', 'Kenza Alaoui', 'Hamza Berrada']
VALID
   {'name': 'Amine El Fassi', 'email': 'amine.elfassi@univ-cadi.ac.ma', 'phone': '+212612345678'}
   {'name': 'Salma Benali', 'email': 'salma_benali@gmail.com', 'phone': '+212661234567'}
   {'name': 'Yassine Idrissi', 'email': 'y.idrissi@outlook.fr', 'phone': '+212661234567'}
   {'name': "Lina O'Brien", 'email': 'lina.obrien@my-startup.io', 'phone': '+212698765432'}
   {'name': 'Kenza Alaoui', 'email': 'kenza.alaoui@gmail.com', 'phone': '+212711223344'}
   {'name': 'Hamza Berrada', 'email': 'hamza@univ-cadi.ac.ma', 'phone': '+212700112233'}
REJECTED
   John Smith -> invalid email
   Bad Entry -> invalid email, invalid phone
   Omar Tazi -> invalid email
Amine El Fassi   a***@univ-cadi.ac.ma
Salma Benali     s***@gmail.com
Yassine Idrissi  y***@outlook.fr
Lina O'Brien     l***@my-startup.io
Kenza Alaoui     k***@gmail.com
Hamza B

---
## Exercice 3 : Nettoyage et transformation d'un article brut (`exo3_article.html`)
### Objectifs :
- Lecture du fichier HTML brut.
- Suppression des commentaires HTML (multi-lignes avec `re.DOTALL` non gourmand).
- Suppression des balises HTML et normalisation des espaces / lignes vides.
- Déduplication des mots consécutifs (`The the`, `was was`) avec rétro-référence et `IGNORECASE`.
- Standardisation des dates au format ISO `YYYY-MM-DD` avec un seul `re.sub` et callback.
- Extraction des montants (`USD` et `EUR`) avec named groups et conversion `float`.
- Extraction des pourcentages avec lookahead et écriture dans `exo3_article_clean.txt`.


In [3]:
import re

# 1. Lecture complète
with open("./files/exo3_article.html", "r", encoding="utf-8") as f:
    html_text = f.read()

# 2. Suppression des commentaires HTML (y compris multi-lignes)
no_comments = re.sub(r"<!--.*?-->", "", html_text, flags=re.DOTALL)

# 3. Suppression des balises HTML et normalisation des espaces
# Remplacement des balises fermantes/ouvrantes de blocs par des sauts de ligne si besoin
text_no_tags = re.sub(r"<[^>]+>", "\n", no_comments)

# Découpage en lignes, compression des espaces multiples en un seul, suppression lignes vides
cleaned_lines = []
for line in text_no_tags.splitlines():
    line = re.sub(r"[ \t]+", " ", line).strip()
    if line:
        cleaned_lines.append(line)

text_cleaned = "\n".join(cleaned_lines)

# 4. Suppression des mots dupliqués consécutifs (ex: 'The the' -> 'The', 'was was' -> 'was')
# \b(\w+)\s+\1\b en re.IGNORECASE, on conserve le premier mot (\1)
text_dedup = re.sub(r"\b(\w+)\s+\1\b", r"\1", text_cleaned, flags=re.IGNORECASE)

# 5. Conversion des dates au format ISO YYYY-MM-DD avec UN SEUL re.sub et callback
months = {
    "january": "01", "february": "02", "march": "03", "april": "04",
    "may": "05", "june": "06", "july": "07", "august": "08",
    "september": "09", "october": "10", "november": "11", "december": "12"
}

def normalize_date(m):
    # Cas 1: Déjà ISO (YYYY-MM-DD) -> inchangé
    if m.group("iso"):
        return m.group("iso")
    # Cas 2: Format DD/MM/YYYY
    elif m.group("dmy"):
        d, mon, y = m.group("d_day"), m.group("d_mon"), m.group("d_yr")
        return f"{y}-{int(mon):02d}-{int(d):02d}"
    # Cas 3: Format D Month YYYY (ex: 7 March 2016)
    elif m.group("text_date"):
        d = int(m.group("t_day"))
        mon_str = m.group("t_mon").lower()
        mon = months[mon_str]
        y = m.group("t_yr")
        return f"{y}-{mon}-{d:02d}"

DATE_PATTERN = re.compile(
    r"""
    (?P<iso>\b\d{4}-\d{2}-\d{2}\b)
    |
    (?P<dmy>\b(?P<d_day>\d{1,2})/(?P<d_mon>\d{1,2})/(?P<d_yr>\d{4})\b)
    |
    (?P<text_date>\b(?P<t_day>\d{1,2})\s+(?P<t_mon>[A-Za-z]+)\s+(?P<t_yr>\d{4})\b)
    """,
    re.VERBOSE
)

def safe_normalize_date(m):
    if m.group("text_date") and m.group("t_mon").lower() not in months:
        return m.group(0)
    return normalize_date(m)

final_text = DATE_PATTERN.sub(safe_normalize_date, text_dedup)

# 6. Extraction des montants ($1,299,450.75 et 45,300.50 EUR)
# Pattern avec named groups usd et eur, et lookarounds pour éviter les correspondances partielles
CURRENCY_PATTERN = re.compile(
    r"""
    (?<!\S)
    (?:
        \$(?P<usd>\d{1,3}(?:,\d{3})*(?:\.\d+)?)(?=[.,;]?(?:\s|$))
        |
        (?P<eur>\d{1,3}(?:,\d{3})*(?:\.\d+)?)\s*EUR(?=[.,;]?(?:\s|$))
    )
    """,
    re.VERBOSE
)

amounts = []
for m in CURRENCY_PATTERN.finditer(final_text):
    if m.group("usd"):
        val = float(m.group("usd").replace(",", ""))
        amounts.append(("USD", val))
    elif m.group("eur"):
        val = float(m.group("eur").replace(",", ""))
        amounts.append(("EUR", val))

# 7. Extraction des pourcentages avec lookahead sur %
percentages = [int(p) for p in re.findall(r"\b(\d+)(?=%)", final_text)]

# Sauvegarde du texte nettoyé dans exo3_article_clean.txt
with open("exo3_article_clean.txt", "w", encoding="utf-8") as f:
    f.write(final_text + "\n")

# Affichage du résultat attendu
print(final_text)
print(f"Amounts : {amounts}")
print(f"Percentages: {percentages}")


Annual Report 2025
The company was founded on 2015-11-03 and opened its first office in Casablanca on 2016-03-07.
In 2025, revenue reached $1,299,450.75, up from $980,200.00 in 2024. Operating costs were 45,300.50 EUR.
The next board meeting is scheduled for 2026-04-18 and the annual party is on 2026-12-21.
Customer satisfaction was very high, reaching 94% in Q4, compared to 88% in Q3.
Amounts : [('USD', 1299450.75), ('USD', 980200.0), ('EUR', 45300.5)]
Percentages: [94, 88]


---
## Exercice 4 : De blocs de texte vers des fichiers CSV (`exo4_orders.txt`)
### Objectifs :
- Extraction multi-lignes de blocs de commandes complètes (`re.VERBOSE | re.MULTILINE`) avec `order_id`, `date`, `customer`, `email`, `items`, `status`.
- Parsing de la section `items` (`qty`, `product`, `unit_price`, calcul de `line_total` et total commande).
- Export vers `orders.csv` et `order_items.csv` (gestion correcte des guillemets comme `27" Monitor`).
- Relecture avec `csv.DictReader` : comptage des commandes par statut et chiffre d'affaires hors commandes annulées.


In [4]:
import re
import csv

# 1. Lecture complète de exo4_orders.txt
with open("./files/exo4_orders.txt", "r", encoding="utf-8") as f:
    orders_text = f.read()

# 2. Pattern pour matcher un bloc complet de commande
ORDER_BLOCK_PATTERN = re.compile(
    r"""
    ^===\s+ORDER\s+\#(?P<order_id>\d+)\s+===\s*\n
    Date:\s*(?P<date>\d{4}-\d{2}-\d{2})\s*\n
    Customer:\s*(?P<customer>[^<\n]+?)(?:\s*<(?P<email>[^>]+)>)?\s*\n
    Items:\s*\n
    (?P<items>(?:[ ]+-\s+.*(?:\n|$))+)
    Status:\s*(?P<status>[A-Z]+)
    """,
    re.VERBOSE | re.MULTILINE
)

# 3. Pattern pour chaque ligne de produit dans items
ITEM_LINE_PATTERN = re.compile(
    r"""
    ^[ ]+-\s+(?P<qty>\d+)\s+x\s+(?P<product>.+?)\s+@\s+(?P<unit_price>\d+(?:\.\d+)?)$
    """,
    re.VERBOSE | re.MULTILINE
)

orders = []
all_order_items = []

matches = list(ORDER_BLOCK_PATTERN.finditer(orders_text))
print(f"{len(matches)} orders found")

for match in matches:
    order_dict = match.groupdict()
    order_id = order_dict["order_id"]
    date = order_dict["date"]
    customer = order_dict["customer"].strip()
    email = order_dict["email"] if order_dict["email"] else ""
    status = order_dict["status"].strip()
    items_block = order_dict["items"]

    parsed_items = []
    order_total = 0.0

    for item_m in ITEM_LINE_PATTERN.finditer(items_block):
        qty = int(item_m.group("qty"))
        product = item_m.group("product").strip()
        unit_price = float(item_m.group("unit_price"))
        line_total = round(qty * unit_price, 2)
        order_total += line_total

        all_order_items.append({
            "order_id": order_id,
            "qty": qty,
            "product": product,
            "unit_price": unit_price,
            "line_total": line_total
        })
        parsed_items.append(product)

    orders.append({
        "order_id": order_id,
        "date": date,
        "customer": customer,
        "email": email,
        "status": status,
        "nb_items": len(parsed_items),
        "total": round(order_total, 2)
    })

# 4. Écriture dans orders.csv
orders_csv_fields = ["order_id", "date", "customer", "email", "status", "nb_items", "total"]
with open("orders.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=orders_csv_fields)
    writer.writeheader()
    writer.writerows(orders)

# 5. Écriture dans order_items.csv
items_csv_fields = ["order_id", "qty", "product", "unit_price", "line_total"]
with open("order_items.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=items_csv_fields)
    writer.writeheader()
    writer.writerows(all_order_items)

# Affichage du contenu des CSV pour vérification
print("\n--- orders.csv ---")
with open("orders.csv", "r", encoding="utf-8") as f:
    print(f.read().strip())

print("\n--- order_items.csv ---")
with open("order_items.csv", "r", encoding="utf-8") as f:
    print(f.read().strip())

# 6. Relecture avec csv.DictReader et statistiques
status_counts = {}
total_revenue_non_cancelled = 0.0

with open("orders.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        st = row["status"]
        status_counts[st] = status_counts.get(st, 0) + 1
        if st != "CANCELLED":
            total_revenue_non_cancelled += float(row["total"])

print(f"\nOrders per status: {status_counts}")
print(f"Revenue (without cancelled orders): {total_revenue_non_cancelled:.2f}")


4 orders found

--- orders.csv ---
order_id,date,customer,email,status,nb_items,total
1042,2026-02-03,Amine El Fassi,amine@example.com,SHIPPED,2,54.48
1043,2026-02-04,Salma Benali,salma.benali@mail.ma,DELIVERED,3,116.4
1044,2026-02-06,Guest User,,CANCELLED,1,36.25
1045,2026-02-09,Lina O'Brien,lina@my-startup.io,PENDING,2,339.0

--- order_items.csv ---
order_id,qty,product,unit_price,line_total
1042,2,USB-C Cable,9.99,19.98
1042,1,Laptop Stand,34.5,34.5
1043,1,Mechanical Keyboard,89.0,89.0
1043,3,Notebook A5,2.5,7.5
1043,1,Desk Lamp,19.9,19.9
1044,5,HDMI Cable,7.25,36.25
1045,1,"27"" Monitor",249.0,249.0
1045,2,Webcam HD,45.0,90.0

Orders per status: {'SHIPPED': 1, 'DELIVERED': 1, 'CANCELLED': 1, 'PENDING': 1}
Revenue (without cancelled orders): 509.88
